# 03 - Quantum-Assisted Navigation with Variational Quantum Circuits (VQC)
This notebook demonstrates quantum angle feature encoding, building parameterized Qiskit quantum circuits, computing expectation values and basis probabilities, and executing quantum-assisted navigation.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils.io_utils import load_json
from simulation.environment import UAVEnvironment
from quantum.feature_encoding import QuantumFeatureEncoder
from quantum.quantum_circuit import ParameterizedNavigationCircuit
from quantum.quantum_model import VariationalQuantumNavigator
from quantum.quantum_decision import QuantumDecisionMaker
from navigation.controller import NavigationController
from dashboard.visualization import plot_trajectory_comparison


## 1. Feature Encoding into Qubit Rotation Angles


In [ ]:
map_data = load_json(PROJECT_ROOT / "data" / "maps" / "map_01.json")
scenario = {"start": [5.0, 5.0], "target": [45.0, 45.0], "max_steps": 100}

env = UAVEnvironment(map_data=map_data, scenario_data=scenario)
state = env.reset()

encoder = QuantumFeatureEncoder()
dummy_features = np.linspace(0.1, 0.9, 12)
rotation_angles = encoder.encode(dummy_features)
print(f"Encoded into {len(rotation_angles)} qubit rotation angles in [-pi, pi]:")
for q, angle in enumerate(rotation_angles):
    print(f"  Qubit q{q}: {angle:+.4f} radians ({np.degrees(angle):+.1f}°)")


## 2. Inspecting the Parameterized Quantum Circuit (Qiskit 2.x)


In [ ]:
circuit_builder = ParameterizedNavigationCircuit()
print("Circuit Qubits:", circuit_builder.circuit.num_qubits)
print("Variational Parameter Count:", circuit_builder.get_num_weights())
print("\nCircuit Diagram:")
print(circuit_builder.circuit.draw(output='text'))


## 3. Simulating the Variational Quantum Navigator


In [ ]:
vqc_model = VariationalQuantumNavigator()
out = vqc_model.forward(encoded_angles=rotation_angles, target_distance=state["target_distance"])

print("Pauli-Z Expectation Values <Z_i>:")
for q, exp_val in enumerate(out["expectations"]):
    print(f"  <Z_{q}>: {exp_val:+.4f}")

print("\nSample Computational Basis State Probabilities:")
for bitstr, prob in list(out["probs_dict"].items())[:4]:
    print(f"  |{bitstr}>: {prob:.4f}")

print("\nTop Action Probabilities:")
for action, prob in sorted(out["action_probabilities"].items(), key=lambda x: x[1], reverse=True)[:4]:
    print(f"  {action:<12}: {prob:.2%}")


## 4. Running Full Quantum-Assisted Navigation Mission


In [ ]:
q_decision = QuantumDecisionMaker()
q_controller = NavigationController(environment=env, navigator=q_decision)
q_controller.reset()

q_result = q_controller.run_episode()
print("Quantum Mission Reached:", q_result["success"])
print("Total Steps Taken:", q_result["total_steps"])
print("Path Length Traveled:", q_result["path_length"])
print("Safety Interventions:", q_result["safety_interventions"])

# Plot quantum flight path
fig = plot_trajectory_comparison(
    scenario=scenario,
    map_data=map_data,
    classical_trajectory=[],
    quantum_trajectory=q_result["trajectory"],
    title="Quantum-Assisted UAV Trajectory",
)
plt.show()
